# Query Optimization Techniques: Filter Pushdown, Pruning, and Join Order

**Module**: 10_performance_and_optimization  
**Notebook**: 03_query_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques to write optimal queries: filter pushdown, column pruning, join reordering, and eliminating redundant computations.
- **Why does this topic exist?**: Writing clean, Catalyst-friendly code prevents unnecessary compute and network overhead.
- **What problem does it solve?**: Queries that scan 50 columns when only 2 are used.
- **Where is it used in real-world Data Engineering?**: Optimizing warehouse queries.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | SQL, joins, partitions, plans |
| **Required depth** | Working knowledge: can apply the main query optimizations and measure their effect |
| **Data Engineer** | Reduce scanned data, shuffled data, and repeated work through query design. |
| **Senior Data Engineer** | Prioritize optimizations by cost and verify them with metrics. |
| **Spark specialist** | Reason about optimizer rules, statistics, and plan stability. |
| **Next topic** | `10_performance_and_optimization/04_shuffle_optimization.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Most query optimization is about doing less work: read fewer columns and partitions, filter earlier, avoid
unnecessary shuffles, and do not repeat expensive computations. Spark's optimizer handles many of these
automatically, but it cannot fix a query that obscures the data or a layout that does not match the access
pattern.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Predicate pushdown** | Filter at the source | Reduces rows read |
| **Column pruning** | Read only needed columns | Reduces I/O and memory |
| **Partition pruning** | Skip directories | Requires filters on partition columns |
| **Join order** | Reorder joins | CBO can choose; broadcast small dimensions |
| **Pre-aggregation** | Reduce before join | Shrinks shuffle input |
| **Statistics** | Table/column metadata | Enables better cost-based decisions |

### How it works, step by step

1. The optimizer parses and analyzes the query.
2. Rules push filters and prune columns.
3. The planner chooses join strategies and orders.
4. AQE may revise the plan after shuffle statistics are available.
5. The executor reads only the surviving data.
6. Metrics show whether the expected reduction happened.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Query_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
# Optimal: Select needed columns and filter early
optimized = orders.filter("status = 'Completed'").select("order_id", "customer_id", "unit_price")
optimized.explain("formatted")

---

## Experiment: Column Pruning I/O Impact

- **1. What are we testing?**: Compare execution time when selecting 2 columns vs all 50 columns.
- **2. What do we expect?**: Column pruning reduces memory deserialization time.

In [ ]:
t0 = time.time(); orders.select("order_id").count(); t_prune = time.time() - t0
t0 = time.time(); orders.count(); t_full = time.time() - t0
print(f"Pruned scan: {t_prune:.3f}s, Full scan: {t_full:.3f}s")

**3. What happened?**: Pruned scan executed faster.  
**4. Why did it happen?**: Only required columns were deserialized.  
**5. Practical Engineering Lesson**: Always select only the specific columns required for downstream operations.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: BI query cost reduction

- **Industry**: Retail
- **Scenario**: A dashboard query scans a wide sales table and joins several dimensions.
- **Problem**: The query selects `*`, filters after aggregation, and shuffles the fact table multiple times.
- **Why this concept is relevant**: Selecting only needed columns, filtering before aggregation, and broadcasting small dimensions cuts I/O and shuffle.
- **Implementation idea**: Rewrite the query, compare scanned bytes and shuffle, and verify the result with row-count checks.
- **Expected result**: A large reduction in runtime and cost with identical output.
- **Alternative**: Create a pre-aggregated summary table.
- **Trade-offs**: A summary table is faster for repeated queries but adds pipeline maintenance and freshness lag.

### Use Case 2: Feature pipeline optimization

- **Industry**: Machine learning
- **Scenario**: A feature engineering job recomputes the same joins for every model.
- **Problem**: Expensive intermediate results are recomputed and shuffled repeatedly.
- **Why this concept is relevant**: Materializing reusable features once reduces repeated work across the pipeline.
- **Implementation idea**: Persist a curated feature table and read it for each model iteration.
- **Expected result**: Faster experimentation and consistent features.
- **Alternative**: Cache the intermediate DataFrame.
- **Trade-offs**: A cache is ephemeral and local; a feature table is durable and shareable.

---

## Business Value

- Lower compute and storage cost.
- Faster dashboards and pipelines.
- More stable performance as data grows.
- Clear optimization priorities for engineers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Filter and project before joins and aggregations.
- Keep statistics fresh for cost-based decisions.
- Broadcast small dimensions; do not force broadcast for large ones.
- Avoid `select('*')` in production queries.
- Measure scanned bytes and shuffle, not only runtime.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Optimize one query at a time

**Steps**

1. Record baseline runtime, scanned bytes, and shuffle for a query.
2. Apply projection and early filtering, then measure.
3. Broadcast a small dimension, then measure again.

**What to observe**: Each change has a distinct effect; combining them without measurement hides which one mattered.

**Healthy result**: The optimized query produces the same result with lower cost.

**Problematic result**: Runtime improves but the result changes because a filter was applied incorrectly.

**Engineering takeaway**: Correctness checks are part of optimization.

---

## Trade-offs and Alternatives

### Advantages

- Large cost reductions
- Mostly automatic
- Composable techniques

### Limitations

- Some queries block pushdown
- Statistics can be stale
- Over-optimization can hurt readability

### When to use it

- BI queries
- ETL pipelines
- Feature engineering

### When not to use it

- Premature micro-optimization
- Hints without measurement
- Ignoring data layout

### Alternatives inside Spark

- Catalyst
- AQE
- Broadcast joins
- Partition pruning

### Alternatives outside Spark

- Warehouse tuning
- Materialized views
- Query accelerators

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter Before Join
**Problem Statement**: Filter table prior to join.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Drop Unused Columns
**Problem Statement**: Drop channel and discount.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Join Order Optimization
**Problem Statement**: How join order affects broadcast decisions.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The optimization that changed the answer

**Scenario**: A join is moved before a filter and the row count changes.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the join type and filter semantics.
2. Verify null handling and duplicate keys.
3. Compare results on a small dataset before and after the rewrite.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter Before Join):
```python
print('Filter orders before joining with customers.')
```

#### Solution 2 (Drop Unused Columns):
```python
orders.drop('channel','discount').show(2)
```

#### Solution 3 (Join Order Optimization):
```python
print('Catalyst CBO can reorder joins if table statistics are analyzed.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Optimization Rules**: Select only needed columns; filter as early as possible; broadcast small tables.

### Essential Technical Terms
- `Column Pruning`: Skipping unused columns.
- `Predicate Pushdown`: Pushing filters to storage.

### Common Engineering Mistakes
- **SELECT * everywhere**: Forces full deserialization of every column.

### Interview Questions on This Topic
- **Q: Why is filtering before a join beneficial?**
  - *A*: It minimizes the number of rows participating in the join and shuffle phases.


### More Interview Questions

- **Q: What is predicate pushdown?**
  - *A*: Moving a filter as close to the data source as possible so fewer rows are read and processed.
- **Q: Why can selecting fewer columns speed up a query?**
  - *A*: Columnar formats read only the referenced columns, reducing I/O, decompression, and memory use.

### What to Study Next
Open 04_shuffle_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')